# RAG Quarterly Earnings Novelty Analysis 
#How to use? Change the ticker and company name and email and use Claude API.

Complete standalone pipeline:
1. Pull the last N earnings 8-K filings from SEC EDGAR
2. Clean and chunk the press release text
3. Derive quarter labels from filing dates (works for any ticker)
4. Embed every chunk with a sentence transformer
5. Score per-chunk novelty vs. the prior quarter (TREC-style)
6. Report the top novel chunks and summary stats

#References for getting inspirations: 
- Allan, Bolivar & Wade (2003)
- Saha, A., et al. (2025). BlackRock LLM agents survey.
- Ghafouri, Mohammadreza, et al. Risk, Ambiguity, and Infinity: Behavioral Signatures of Modern Large Language Models. 2025.
- Allan, James, Courtney Wade, and Alvaro Bolivar. Retrieval and Novelty Detection at the Sentence Level. 2003. 

# Actual implementations: 
- Karpukhin et al. (2020), EMNLP — Dense Passage Retrieval
- Soboroff & Harman (2005), NIST — TREC Novelty Track

## 1. Install dependencies

In [1]:
# Run once. Comment out lines you already have.
!pip install -U edgartools sentence-transformers scikit-learn pandas -q

#!pip install sentence-transformers
#!pip install scikit-learn
#!pip install pandas
#!pip install anthropic
#!pip install nltk
#!pip install numpy

In [2]:
#Set email and company name. 

In [ ]:
# ── Config (interactive) ─────────────────────────────────────────

# ── Company name first ───────────────────────────────────────────
COMPANY_NAME = input("Company name (e.g., Microsoft, Apple, NVIDIA, Google): ").strip()

# ── Ticker ────────────────────────────────────────────────────────
TICKER = input(
    "Stock ticker (e.g., MSFT, AAPL, NVDA, GOOGL, META, AMZN, TSLA, CRM): "
).strip().upper()

# ── SEC email ─────────────────────────────────────────────────────
SEC_EMAIL = input("Your email (required by SEC EDGAR): ").strip()

# ── Fiscal year end month ─────────────────────────────────────────
fy_input = input(
    "Fiscal year end month (1-12). "
    "MSFT=6, AAPL=9, NVDA=1, CRM=1, ORCL=5, ADBE=11, COST=8. "
    "Most others (GOOGL, META, AMZN, TSLA, AMD, INTC, IBM) = 12. "
    "Press Enter for 12: "
).strip()
FISCAL_YEAR_END_MONTH = int(fy_input) if fy_input else 12

# ── Number of quarters ────────────────────────────────────────────
n_input = input("How many quarters of earnings? [default 5]: ").strip()
MAX_EARNINGS = int(n_input) if n_input else 5
MAX_SCAN = MAX_EARNINGS * 6   # scan ~6x as many 8-Ks (most aren't earnings)

# ── Chunking parameters — rarely need changing ───────────────────
MAX_TOKENS          = 400
OVERLAP_SENTENCES   = 1
MIN_CHUNK_CHARS     = 100
AVG_CHARS_PER_TOKEN = 4

# ── Confirmation ──────────────────────────────────────────────────
month_names = ["", "Jan", "Feb", "Mar", "Apr", "May", "Jun",
                   "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
print("\n" + "="*50)
print(f"  Configured:")
print(f"    Company:       {COMPANY_NAME}")
print(f"    Ticker:        {TICKER}")
print(f"    SEC email:     {SEC_EMAIL}")
print(f"    Fiscal Y/E:    {month_names[FISCAL_YEAR_END_MONTH]} (month {FISCAL_YEAR_END_MONTH})")
print(f"    # Quarters:    {MAX_EARNINGS}")
print("="*50)

## 2. Configuration

Set the ticker, your contact email (SEC requires it), and how many quarters of earnings to pull.

In [207]:
import pandas as pd
from edgar import Company, set_identity

set_identity(SEC_EMAIL)

# ─── Extract text from a filing object ───────────────────────────
def extract_text_from_filing(filing_obj) -> str:
    """Try multiple methods to get text from a filing object."""

    # method 1: press release text (best for 8-K earnings)
    try:
        prs = filing_obj.press_releases
        if prs and len(prs) > 0:
            pr = prs[0]
            for attr in ["text", "markdown", "content", "html"]:
                if hasattr(pr, attr):
                    val = getattr(pr, attr)
                    if callable(val):
                        try:
                            result = val()
                            if isinstance(result, str) and len(result) > 200:
                                return result[:50000]
                        except Exception:
                            pass
                    elif isinstance(val, str) and len(val) > 200:
                        return val[:50000]
    except Exception:
        pass

    # method 2: filing .text() method
    try:
        text = filing_obj.text()
        if isinstance(text, str) and len(text) > 200:
            return text[:50000]
    except Exception:
        pass

    # method 3: filing .markdown() method
    try:
        text = filing_obj.markdown()
        if isinstance(text, str) and len(text) > 200:
            return text[:50000]
    except Exception:
        pass

    # method 4: str fallback
    try:
        text = str(filing_obj)
        if len(text) > 200:
            return text[:50000]
    except Exception:
        pass

    return ""

# ─── Check if an 8-K is an earnings filing ───────────────────────
def is_earnings_filing(filing_obj) -> bool:
    """Item 2.02 = Results of Operations and Financial Condition."""
    try:
        items = filing_obj.items
        if any("2.02" in str(item) for item in items):
            return True
    except Exception:
        pass

    try:
        filing_str = str(filing_obj).lower()
        if any(kw in filing_str for kw in [
            "2.02", "results of operations",
            "quarterly results", "financial results",
            "earnings per share", "revenue was"
        ]):
            return True
    except Exception:
        pass

    return False

# ─── Pull N quarters of earnings 8-K ─────────────────────────────
def get_earnings_filings(ticker: str, company_name: str,
                          max_earnings: int = MAX_EARNINGS,
                          max_scan: int = MAX_SCAN) -> pd.DataFrame:
    print(f"\n{'='*60}")
    print(f"  EDGAR EARNINGS COLLECTION — {ticker}")
    print(f"  Target: last {max_earnings} earnings quarters")
    print(f"{'='*60}")

    company = Company(ticker)
    print(f"  ✓ Found: {company.name}")

    filings = company.get_filings(form="8-K").head(max_scan)
    results = []

    for filing in filings:
        try:
            filing_date = str(filing.filing_date)
            obj = filing.obj()

            if not is_earnings_filing(obj):
                continue

            text = extract_text_from_filing(obj)
            if not text or len(text.strip()) < 200:
                continue

            results.append({
                "filing_date": filing_date,
                "ticker":      ticker,
                "company":     company_name,
                "form":        "8-K",
                "text":        text,
            })
            print(f"  ✓ {filing_date}  ({len(text):,} chars)")

            if len(results) >= max_earnings:
                break
        except Exception as e:
            print(f"  ! skipped {filing.filing_date}: {e}")
            continue

    df = pd.DataFrame(results)
    print(f"\n  Collected {len(df)} earnings filings")
    return df

df_edgar = get_earnings_filings(TICKER, COMPANY_NAME)
df_edgar[['filing_date', 'ticker']]


  EDGAR EARNINGS COLLECTION — MSFT
  Target: last 5 earnings quarters
  ✓ Found: MICROSOFT CORP
  ✓ 2026-01-28  (36,460 chars)
  ✓ 2025-10-29  (32,755 chars)
  ✓ 2025-07-30  (34,021 chars)
  ✓ 2025-04-30  (33,438 chars)
  ✓ 2025-01-29  (32,659 chars)

  Collected 5 earnings filings


,filing_date,ticker
0,2026-01-28,MSFT
1,2025-10-29,MSFT
2,2025-07-30,MSFT
3,2025-04-30,MSFT
4,2025-01-29,MSFT


## 3. Pull earnings 8-K filings from SEC EDGAR

In [208]:
import pandas as pd
from edgar import Company, set_identity

set_identity(SEC_EMAIL)

# ─── Extract text from a filing object ───────────────────────────
def extract_text_from_filing(filing_obj) -> str:
    """Try multiple methods to get text from a filing object."""

    # method 1: press release text (best for 8-K earnings)
    try:
        prs = filing_obj.press_releases
        if prs and len(prs) > 0:
            pr = prs[0]
            for attr in ["text", "markdown", "content", "html"]:
                if hasattr(pr, attr):
                    val = getattr(pr, attr)
                    if callable(val):
                        try:
                            result = val()
                            if isinstance(result, str) and len(result) > 200:
                                return result[:50000]
                        except Exception:
                            pass
                    elif isinstance(val, str) and len(val) > 200:
                        return val[:50000]
    except Exception:
        pass

    # method 2: filing .text() method
    try:
        text = filing_obj.text()
        if isinstance(text, str) and len(text) > 200:
            return text[:50000]
    except Exception:
        pass

    # method 3: filing .markdown() method
    try:
        text = filing_obj.markdown()
        if isinstance(text, str) and len(text) > 200:
            return text[:50000]
    except Exception:
        pass

    # method 4: str fallback
    try:
        text = str(filing_obj)
        if len(text) > 200:
            return text[:50000]
    except Exception:
        pass

    return ""

# ─── Check if an 8-K is an earnings filing ───────────────────────
def is_earnings_filing(filing_obj) -> bool:
    """Item 2.02 = Results of Operations and Financial Condition."""
    try:
        items = filing_obj.items
        if any("2.02" in str(item) for item in items):
            return True
    except Exception:
        pass

    try:
        filing_str = str(filing_obj).lower()
        if any(kw in filing_str for kw in [
            "2.02", "results of operations",
            "quarterly results", "financial results",
            "earnings per share", "revenue was"
        ]):
            return True
    except Exception:
        pass

    return False

# ─── Pull N quarters of earnings 8-K ─────────────────────────────
def get_earnings_filings(ticker: str, company_name: str,
                          max_earnings: int = MAX_EARNINGS,
                          max_scan: int = MAX_SCAN) -> pd.DataFrame:
    print(f"\n{'='*60}")
    print(f"  EDGAR EARNINGS COLLECTION — {ticker}")
    print(f"  Target: last {max_earnings} earnings quarters")
    print(f"{'='*60}")

    company = Company(ticker)
    print(f"  ✓ Found: {company.name}")

    filings = company.get_filings(form="8-K").head(max_scan)
    results = []

    for filing in filings:
        try:
            filing_date = str(filing.filing_date)
            obj = filing.obj()

            if not is_earnings_filing(obj):
                continue

            text = extract_text_from_filing(obj)
            if not text or len(text.strip()) < 200:
                continue

            results.append({
                "filing_date": filing_date,
                "ticker":      ticker,
                "company":     company_name,
                "form":        "8-K",
                "text":        text,
            })
            print(f"  ✓ {filing_date}  ({len(text):,} chars)")

            if len(results) >= max_earnings:
                break
        except Exception as e:
            print(f"  ! skipped {filing.filing_date}: {e}")
            continue

    df = pd.DataFrame(results)
    print(f"\n  Collected {len(df)} earnings filings")
    return df

df_edgar = get_earnings_filings(TICKER, COMPANY_NAME)
df_edgar[['filing_date', 'ticker']]


  EDGAR EARNINGS COLLECTION — MSFT
  Target: last 5 earnings quarters
  ✓ Found: MICROSOFT CORP
  ✓ 2026-01-28  (36,460 chars)
  ✓ 2025-10-29  (32,755 chars)
  ✓ 2025-07-30  (34,021 chars)
  ✓ 2025-04-30  (33,438 chars)
  ✓ 2025-01-29  (32,659 chars)

  Collected 5 earnings filings


,filing_date,ticker
0,2026-01-28,MSFT
1,2025-10-29,MSFT
2,2025-07-30,MSFT
3,2025-04-30,MSFT
4,2025-01-29,MSFT


## 4. Derive quarter labels from filing dates

No hardcoded `QUARTER_MAP`. Uses fiscal-year-end month to produce labels like `FQ1-2026`.

In [209]:
from datetime import datetime

def filing_date_to_quarter(filing_date: str,
                            fy_end_month: int = FISCAL_YEAR_END_MONTH) -> str:
    """
    Approximate fiscal quarter label from filing date.
    A 10-Q/8-K earnings filing typically comes ~1 month after the
    fiscal quarter ends, so we back off one month before bucketing.
    """
    d = datetime.strptime(str(filing_date)[:10], "%Y-%m-%d")
    # shift back ~30 days so the filing date maps to the reporting quarter
    reporting_month = d.month - 1 if d.month > 1 else 12
    reporting_year  = d.year if d.month > 1 else d.year - 1

    # months since fiscal year end (modulo 12)
    months_into_fy = (reporting_month - fy_end_month - 1) % 12
    fq = months_into_fy // 3 + 1

    # fiscal year containing this reporting month
    if reporting_month > fy_end_month:
        fy = reporting_year + 1
    else:
        fy = reporting_year

    return f"FQ{fq}-{fy}"

# Apply to df_edgar for sanity check
df_edgar['quarter'] = df_edgar['filing_date'].apply(filing_date_to_quarter)

print("Quarter derivation check:")
print(df_edgar[['filing_date', 'quarter']].to_string(index=False))
print("\nIf the quarter labels look wrong, adjust FISCAL_YEAR_END_MONTH in Section 2.")

Quarter derivation check:
filing_date  quarter
 2026-01-28 FQ2-2026
 2025-10-29 FQ1-2026
 2025-07-30 FQ4-2025
 2025-04-30 FQ3-2025
 2025-01-29 FQ2-2025

If the quarter labels look wrong, adjust FISCAL_YEAR_END_MONTH in Section 2.


## 5. Clean and chunk the earnings text

In [210]:
import re
import nltk
from nltk.tokenize import sent_tokenize
from dataclasses import dataclass, field

nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)

# Strip patterns for earnings boilerplate
STRIP_PATTERNS_EARNINGS = [
    r"(?i)forward.looking statements.{0,2000}",
    r"(?i)safe harbor.{0,1000}",
    r"(?i)private securities litigation.{0,500}",
    r"(?i)(investor relations|media contact|for more information).{0,300}",
    r"(?i)^(document|exhibit\s*\d+\.\d+)\s*$",
]

@dataclass
class Chunk:
    text:         str
    company:      str
    ticker:       str
    source_type:  str
    filing_date:  str
    quarter:      str
    chunk_index:  int
    chunk_total:  int
    char_count:   int = field(init=False)
    token_count:  int = field(init=False)

    def __post_init__(self):
        self.char_count  = len(self.text)
        self.token_count = len(self.text) // AVG_CHARS_PER_TOKEN

def clean_text(text: str) -> str:
    if not text:
        return ""
    # strip HTML
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"&[a-z]+;", " ", text)
    # strip legal boilerplate
    for pattern in STRIP_PATTERNS_EARNINGS:
        text = re.sub(pattern, "", text, flags=re.DOTALL | re.MULTILINE)
    # drop lines that are mostly numbers (financial table rows)
    lines, clean_lines = text.split('\n'), []
    for line in lines:
        stripped = line.strip()
        alpha    = sum(1 for c in stripped if c.isalpha())
        total    = len(stripped)
        if total == 0 or alpha / total >= 0.4:
            clean_lines.append(line)
    text = '\n'.join(clean_lines)
    # normalize whitespace
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]{2,}', ' ', text)
    return text.strip()

def semantic_chunk(text: str, max_tokens: int = MAX_TOKENS,
                    overlap: int = OVERLAP_SENTENCES) -> list:
    try:
        sentences = sent_tokenize(text)
    except Exception:
        sentences = re.split(r'(?<=[.!?])\s+', text)
    if not sentences:
        return []

    chunks, current, cur_tokens = [], [], 0
    for sent in sentences:
        sent_tokens = len(sent) // AVG_CHARS_PER_TOKEN
        if cur_tokens + sent_tokens > max_tokens and current:
            chunks.append(' '.join(current))
            current = current[-overlap:] if overlap > 0 else []
            cur_tokens = sum(len(s) // AVG_CHARS_PER_TOKEN for s in current)
        current.append(sent)
        cur_tokens += sent_tokens
    if current:
        chunks.append(' '.join(current))
    return [c for c in chunks if len(c) >= MIN_CHUNK_CHARS]

def chunk_earnings_report(raw_text: str, company: str, ticker: str,
                           filing_date: str, quarter: str) -> list:
    cleaned = clean_text(raw_text)
    if not cleaned:
        return []
    pieces = semantic_chunk(cleaned)
    total = len(pieces)
    return [
        Chunk(
            text=p, company=company, ticker=ticker,
            source_type='earnings', filing_date=filing_date,
            quarter=quarter, chunk_index=i, chunk_total=total,
        )
        for i, p in enumerate(pieces)
    ]

# ── Chunk all quarters ───────────────────────────────────────────
all_chunks = []
for _, row in df_edgar.iterrows():
    chunks = chunk_earnings_report(
        raw_text=row['text'], company=row['company'], ticker=row['ticker'],
        filing_date=row['filing_date'], quarter=row['quarter'],
    )
    all_chunks.extend(chunks)
    print(f"  ✓ {row['quarter']} ({row['filing_date']}): {len(chunks)} chunks")

df_earnings = pd.DataFrame([c.__dict__ for c in all_chunks])
print(f"\nTotal chunks: {len(df_earnings)}")
df_earnings.head(3)

  ✓ FQ2-2026 (2026-01-28): 10 chunks
  ✓ FQ1-2026 (2025-10-29): 9 chunks
  ✓ FQ4-2025 (2025-07-30): 8 chunks
  ✓ FQ3-2025 (2025-04-30): 8 chunks
  ✓ FQ2-2025 (2025-01-29): 8 chunks

Total chunks: 43


,text,company,ticker,source_type,filing_date,quarter,chunk_index,chunk_total,char_count,token_count
0,Microsoft Cloud and AI Strength Drives Second ...,Microsoft,MSFT,earnings,2026-01-28,FQ2-2026,0,10,1374,343
1,“We are pushing the frontier across our entire...,Microsoft,MSFT,earnings,2026-01-28,FQ2-2026,1,10,1077,269
2,"Three Months Ended December 31,\n($ in million...",Microsoft,MSFT,earnings,2026-01-28,FQ2-2026,2,10,1810,452


## 6. Content novelty analysis (Cell A)

For each chunk in quarter N, novelty = `1 - max_cosine_similarity` to any chunk in quarter N-1.

In [211]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# ── 1. Embed every chunk ─────────────────────────────────────────
print("Loading embedder (all-MiniLM-L6-v2)...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

df_earnings = df_earnings.reset_index(drop=True)

print(f"Embedding {len(df_earnings)} chunks...")
chunk_embeddings = embedder.encode(
    df_earnings['text'].astype(str).tolist(),
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print(f"✓ Embeddings shape: {chunk_embeddings.shape}")

# ── 2. Light boilerplate filter (TREC two-stage pattern) ─────────
BOILERPLATE_KEYWORDS = [
    'forward-looking', 'safe harbor', 'private securities litigation',
    'investor relations contact', 'media contact', 'cautionary statement',
    'this press release contains', 'non-gaap',
]

def is_boilerplate(text: str) -> bool:
    t = str(text).lower()
    if len(t) < 150:
        return True
    hits = sum(1 for kw in BOILERPLATE_KEYWORDS if kw in t)
    return hits >= 2

df_earnings['is_boilerplate'] = df_earnings['text'].apply(is_boilerplate)
n_boiler = df_earnings['is_boilerplate'].sum()
print(f"\nBoilerplate chunks filtered: {n_boiler} / {len(df_earnings)}")

# ── 3. Build chronological quarter order ─────────────────────────
quarter_order = (
    df_earnings[['quarter', 'filing_date']]
    .drop_duplicates().sort_values('filing_date').reset_index(drop=True)
)
print(f"\nQuarter order (oldest → newest):")
for _, row in quarter_order.iterrows():
    print(f"  {row['quarter']}  ({row['filing_date']})")

quarters_list = quarter_order['quarter'].tolist()

# ── 4. Compute novelty per chunk ─────────────────────────────────
novelty_scores    = np.full(len(df_earnings), np.nan)
nearest_prior_idx = np.full(len(df_earnings), -1, dtype=int)

for i in range(1, len(quarters_list)):
    cur_q, prev_q = quarters_list[i], quarters_list[i - 1]
    cur_idx  = df_earnings.index[df_earnings['quarter'] == cur_q].to_numpy()
    prev_idx = df_earnings.index[df_earnings['quarter'] == prev_q].to_numpy()

    if len(cur_idx) == 0 or len(prev_idx) == 0:
        print(f"  skip {prev_q} → {cur_q} (empty quarter)")
        continue

    sim     = cosine_similarity(chunk_embeddings[cur_idx], chunk_embeddings[prev_idx])
    max_sim = sim.max(axis=1)
    argmax  = sim.argmax(axis=1)

    novelty_scores[cur_idx]    = 1.0 - max_sim
    nearest_prior_idx[cur_idx] = prev_idx[argmax]

df_earnings['novelty']           = novelty_scores
df_earnings['nearest_prior_idx'] = nearest_prior_idx

Loading embedder (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding 43 chunks...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

✓ Embeddings shape: (43, 384)

Boilerplate chunks filtered: 2 / 43

Quarter order (oldest → newest):
  FQ2-2025  (2025-01-29)
  FQ3-2025  (2025-04-30)
  FQ4-2025  (2025-07-30)
  FQ1-2026  (2025-10-29)
  FQ2-2026  (2026-01-28)


## 7. Report top novel chunks per quarter

In [212]:
TOP_N = 5
print(f"{'='*72}")
print(f"  TOP {TOP_N} NOVEL CHUNKS PER QUARTER")
print(f"  (content most unlike anything in the previous quarter)")
print(f"{'='*72}")

for i in range(1, len(quarters_list)):
    cur_q, prev_q = quarters_list[i], quarters_list[i - 1]
    q_df = df_earnings[
        (df_earnings['quarter'] == cur_q)
        & (~df_earnings['is_boilerplate'])
        & (df_earnings['novelty'].notna())
    ].sort_values('novelty', ascending=False).head(TOP_N)

    print(f"\n── {cur_q} vs {prev_q} ──")
    if len(q_df) == 0:
        print("  (no substantive chunks with valid novelty)")
        continue
    for rank, (_, row) in enumerate(q_df.iterrows(), 1):
        prior = df_earnings.loc[row['nearest_prior_idx']]
        print(f"\n  [#{rank}] novelty = {row['novelty']:.3f}")
        print(f"  {cur_q}:   {str(row['text'])[:240].strip()}...")
        print(f"  closest {prev_q}: {str(prior['text'])[:200].strip()}...")

  TOP 5 NOVEL CHUNKS PER QUARTER
  (content most unlike anything in the previous quarter)

── FQ3-2025 vs FQ2-2025 ──

  [#1] novelty = 0.072
  FQ3-2025:   Business Highlights
Revenue in Productivity and Business Processes was $29.9 billion and increased 10% (up 13% in constant currency), with the following business highlights:
•Microsoft 365 Commercial products and cloud services revenue incr...
  closest FQ2-2025: “Already, our AI business has surpassed an annual revenue run rate of $13 billion, up 175% year-over-year.”
 “This quarter Microsoft Cloud revenue was $40.9 billion, up 21% year-over-year,” said Amy H...

  [#2] novelty = 0.060
  FQ3-2025:   Microsoft Cloud and AI Strength Drives Third Quarter Results
REDMOND, Wash. — April 30, 2025 — Microsoft Corp. today announced the following results for the quarter ended March 31, 2025, as compared to the corresponding period of last fisca...
  closest FQ2-2025: Microsoft Cloud and AI Strength Drives Second Quarter Results
REDMOND, Was

In [213]:
# ── Latest vs prior quarter — uses results from Section 6 ────────
#TOP_N = 10

# quarters_list is already chronological (oldest → newest) from Section 6
#latest_q = quarters_list[-1]
#prior_q  = quarters_list[-2]

# Filter: latest quarter, substantive only, with valid novelty
#latest_novel = df_earnings[
#    (df_earnings['quarter'] == latest_q)
#    & (~df_earnings['is_boilerplate'])
#    & (df_earnings['novelty'].notna())
#].sort_values('novelty', ascending=False).head(TOP_N)

#print(f"{'='*72}")
#print(f"  WHAT'S NEW IN {latest_q} vs {prior_q}")
#print(f"  Top {TOP_N} most novel chunks")
#print(f"{'='*72}")

#for rank, (_, row) in enumerate(latest_novel.iterrows(), 1):
#    prior_chunk = df_earnings.loc[row['nearest_prior_idx']]
#    print(f"\n[#{rank}] novelty = {row['novelty']:.3f}")
#    print(f"\n{latest_q}:")
#    print(f"  {str(row['text'])[:400].strip()}{'...' if len(row['text']) > 400 else ''}")
#    print(f"\nClosest match in {prior_q}:")
#    print(f"  {str(prior_chunk['text'])[:300].strip()}{'...' if len(prior_chunk['text']) > 300 else ''}")
#    print('-' * 72)

# Quick summary
#print(f"\n{'='*72}")
#print(f"  SUMMARY: {latest_q} vs {prior_q}")
#print(f"{'='*72}")
#substantive = df_earnings[
#    (df_earnings['quarter'] == latest_q)
#    & (~df_earnings['is_boilerplate'])
#    & (df_earnings['novelty'].notna())
#]
#print(f"  Substantive chunks: {len(substantive)}")
#print(f"  Mean novelty:   {substantive['novelty'].mean():.3f}")
#print(f"  Median novelty: {substantive['novelty'].median():.3f}")
#print(f"  Max novelty:    {substantive['novelty'].max():.3f}")

## 8. Summary stats per quarter

In [214]:
summary = (
    df_earnings[~df_earnings['is_boilerplate']]
    .groupby('quarter')['novelty']
    .agg(['count', 'mean', 'median', 'max'])
    .rename(columns={'count': 'n_chunks'})
    .round(3)
)
summary = summary.reindex([q for q in quarters_list if q in summary.index])

print(f"{'='*60}")
print(f"  NOVELTY SUMMARY PER QUARTER")
print(f"{'='*60}")
print(summary.to_string())
print("\nInterpretation:")
print("  - The earliest quarter shows NaN (no prior to compare — correct).")
print("  - High mean novelty → broadly new material vs prior quarter.")
print("  - High max but moderate mean → one specific new topic introduced.")

  NOVELTY SUMMARY PER QUARTER
          n_chunks   mean  median    max
quarter                                 
FQ2-2025         0    NaN     NaN    NaN
FQ3-2025         8  0.031   0.027  0.072
FQ4-2025         8  0.106   0.041  0.243
FQ1-2026         8  0.116   0.103  0.204
FQ2-2026         9  0.038   0.039  0.073

Interpretation:
  - The earliest quarter shows NaN (no prior to compare — correct).
  - High mean novelty → broadly new material vs prior quarter.
  - High max but moderate mean → one specific new topic introduced.


In [215]:
# ── Install if needed ────────────────────────────────────────────
!pip install anthropic -q

In [216]:
# ── API Config (interactive) ─────────────────────────────────────
import os
import getpass

# Use getpass so the key doesn't show on screen as you type/paste
# Falls back to env var if it's already set, so you can avoid retyping
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
if not ANTHROPIC_API_KEY:
    ANTHROPIC_API_KEY = getpass.getpass("Enter your Anthropic API key (sk-ant-...): ").strip()

assert ANTHROPIC_API_KEY.startswith("sk-ant-"), \
    "That doesn't look like an Anthropic key — should start with 'sk-ant-'"

# How many top novel chunks per transition to send to the LLM.
TOP_N_FOR_LLM = 10

# Model choice
MODEL = "claude-sonnet-4-5"

print("✓ API key loaded")
print(f"✓ Using model: {MODEL}")

✓ API key loaded
✓ Using model: claude-sonnet-4-5


In [217]:
# ── Build the quarter-transition pairs for the LLM ───────────────
# For each transition, assemble the top-N novel chunks along with
# their closest prior-quarter neighbor. The LLM will see the diff.
import pandas as pd

def build_transition_payload(df: pd.DataFrame,
                              cur_q: str, prev_q: str,
                              top_n: int = TOP_N_FOR_LLM) -> list:
    """Return a list of {novel_chunk, closest_prior_chunk, novelty} dicts."""
    q_df = df[
        (df['quarter'] == cur_q)
        & (~df['is_boilerplate'])
        & (df['novelty'].notna())
    ].sort_values('novelty', ascending=False).head(top_n)

    pairs = []
    for _, row in q_df.iterrows():
        prior = df.loc[row['nearest_prior_idx']]
        pairs.append({
            'novelty': round(float(row['novelty']), 3),
            'novel_chunk': str(row['text']).strip(),
            'closest_prior_chunk': str(prior['text']).strip(),
        })
    return pairs

# Verify it works by printing the FQ4-2025 payload shape
test_pairs = build_transition_payload(df_earnings, 'FQ4-2025', 'FQ3-2025')
print(f"Built {len(test_pairs)} pairs for FQ3-2025 → FQ4-2025")
print(f"First pair novelty: {test_pairs[0]['novelty']}")
print(f"First novel chunk starts: {test_pairs[0]['novel_chunk'][:100]}...")

Built 8 pairs for FQ3-2025 → FQ4-2025
First pair novelty: 0.243
First novel chunk starts: Fiscal Year 2025 Results
Microsoft Corp. today announced the following results for the fiscal year e...


In [218]:
  # ── The prompt ───────────────────────────────────────────────────
  # Prompt design notes:
  # - Structured JSON output with typed categories reduces editorializing
  # - Require verbatim quotes to ground every claim
  # - Explicit instruction NOT to comment on tone/hedging (that's Cell B)
  # - Neutral framing: "characterize what is new," not "interpret significance"

  SYSTEM_PROMPT = """You are a financial text analyst examining quarter-over-quarter changes in earnings press releases.

  You will receive pairs of text chunks. In each pair:
  - "novel_chunk" is a passage from the current quarter's earnings release
  - "closest_prior_chunk" is the most semantically similar passage from the previous quarter's release
  - "novelty" is a score from 0 to 1 where higher values mean less similar

  Your job: identify what NEW CONTENT appears in the current quarter that wasn't in the previous quarter. Classify each distinct new theme into one of these categories:

  - new_product: A product, service, SKU, or offering mentioned that wasn't in the prior quarter
  - new_metric: A financial or operating metric being reported that wasn't reported before, OR a new segment breakdown
  - new_risk: A risk factor, challenge, or headwind mentioned that wasn't in the prior quarter
  - new_segment: A business segment, customer type, or geography being discussed that wasn't before
  - new_framing: The same topic as last quarter but described in a substantively different way (e.g., new terminology, new emphasis, different framing of the same business)
  - other: New content that doesn't fit the above categories

  STRICT RULES:
  1. Every theme you identify MUST include a verbatim quote from the novel_chunk text. Put quotes in a "quote" field.
  2. Do NOT comment on tone, confidence, hedging, sentiment, or how cautious/bullish the language sounds. Only describe what content is new.
  3. Do NOT speculate about why the change happened or what it means for the business. Only describe what is different.
  4. If a "novel_chunk" is actually similar to its prior chunk (just reworded), say so in the "notes" field and mark it as new_framing with low confidence.
  5. Consolidate themes across multiple chunks — if three chunks all introduce the same new product, output ONE theme with all three quotes.

  Output format: valid JSON only, no other text. Schema:
  {
    "themes": [
      {
        "category": "new_product" | "new_metric" | "new_risk" | "new_segment" | "new_framing" | "other",
        "label": "Short descriptive label (3-8 words)",
        "description": "1-2 sentence neutral description of what's new",
        "quotes": ["verbatim phrase from novel_chunk", "another verbatim phrase"],
        "confidence": "high" | "medium" | "low",
        "notes": "any caveats about whether this is genuinely new"
      }
    ]
  }"""

  def build_user_message(prev_q: str, cur_q: str, pairs: list) -> str:
      """Format the payload as the user message."""
      lines = [
          f"Transition: {prev_q} → {cur_q}",
          f"Number of pairs: {len(pairs)}",
          "",
          "Pairs (sorted by novelty, highest first):",
          "",
      ]
      for i, p in enumerate(pairs, 1):
          lines.append(f"--- Pair {i} (novelty={p['novelty']}) ---")
          lines.append(f"NOVEL [{cur_q}]:")
          lines.append(p['novel_chunk'])
          lines.append("")
          lines.append(f"CLOSEST PRIOR [{prev_q}]:")
          lines.append(p['closest_prior_chunk'])
          lines.append("")
      return "\n".join(lines)

  # Preview what gets sent for one transition
  preview = build_user_message('FQ3-2025', 'FQ4-2025', test_pairs[:2])
  print(preview[:1500])
  print("...(truncated for preview)")



Transition: FQ3-2025 → FQ4-2025
Number of pairs: 2

Pairs (sorted by novelty, highest first):

--- Pair 1 (novelty=0.243) ---
NOVEL [FQ4-2025]:
Fiscal Year 2025 Results
Microsoft Corp. today announced the following results for the fiscal year ended June 30, 2025, as compared to the corresponding period of last fiscal year:
•Revenue was $281.7 billion and increased 15%
•Operating income was $128.5 billion and increased 17% (up 18% in constant currency)
•Net income was $101.8 billion and increased 16% (up 15% in constant currency)
•Diluted earnings per share was $13.64 and increased 16%
Business Outlook
Microsoft will provide forward-looking guidance in connection with this quarterly earnings announcement on its earnings conference call and webcast. Quarterly Highlights, Product Releases, and Enhancements
Every quarter Microsoft delivers hundreds of products, either as new releases, services, or enhancements to current products and services. These releases are a result of significant res

In [219]:
# ── Timeline: latest quarter vs. every prior quarter ─────────────
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

latest_q = quarters_list[-1]
prior_quarters = quarters_list[:-1]  # chronological: oldest → newest

# Build the timeline table
timeline_rows = []
for q in prior_quarters:
    score = nov_matrix.loc[latest_q, q]
    months_back = (len(quarters_list) - 1) - quarters_list.index(q)  # quarters back
    timeline_rows.append({
        'prior_quarter': q,
        'quarters_back': months_back,
        'novelty_vs_latest': round(float(score), 3),
    })

timeline = pd.DataFrame(timeline_rows)

print("="*72)
print(f"  TIMELINE: {latest_q} compared against every prior quarter")
print(f"  (chronological order — oldest first)")
print("="*72)
print(timeline.to_string(index=False))

# Identify extremes
most_diff = timeline.loc[timeline['novelty_vs_latest'].idxmax()]
most_sim  = timeline.loc[timeline['novelty_vs_latest'].idxmin()]

print(f"\n  ▲ MOST DIFFERENT from {latest_q}:")
print(f"     {most_diff['prior_quarter']}  "
      f"(novelty = {most_diff['novelty_vs_latest']:.3f}, "
      f"{int(most_diff['quarters_back'])} quarters back)")

print(f"\n  ▼ MOST SIMILAR to {latest_q}:")
print(f"     {most_sim['prior_quarter']}  "
      f"(novelty = {most_sim['novelty_vs_latest']:.3f}, "
      f"{int(most_sim['quarters_back'])} quarters back)")

# ── ASCII bar chart for quick visual ─────────────────────────────
print(f"\n{'='*72}")
print(f"  VISUAL TIMELINE")
print(f"{'='*72}")
max_score = timeline['novelty_vs_latest'].max()
bar_width = 40
for _, row in timeline.iterrows():
    bar_len = int((row['novelty_vs_latest'] / max_score) * bar_width) if max_score > 0 else 0
    bar = '█' * bar_len
    marker = ''
    if row['prior_quarter'] == most_diff['prior_quarter']:
        marker = '  ← most differen
    elif row['prior_quarter'] == most_sim['prior_quarter']:
        marker = '  ← most similar'
    print(f"  {row['prior_quarter']:>10}  {bar:<{bar_width}}  {row['novelty_vs_latest']:.3f}{marker}")

# ── Show what's driving the biggest gap ──────────────────────────
print(f"\n{'='*72}")
print(f"  TOP 5 PASSAGES IN {latest_q} MOST UNLIKE ANYTHING IN {most_diff['prior_quarter']}")
print(f"{'='*72}")

idx_latest = df_earnings[(df_earnings['quarter'] == latest_q)
                         & (~df_earnings['is_boilerplate'])].index
idx_far = df_earnings[(df_earnings['quarter'] == most_diff['prior_quarter'])
                      & (~df_earnings['is_boilerplate'])].index

sims = cosine_similarity(chunk_embeddings[idx_latest], chunk_embeddings[idx_far])
novelty_per_chunk = 1 - sims.max(axis=1)

ranked = pd.DataFrame({
    'novelty': novelty_per_chunk,
    'text': df_earnings.loc[idx_latest, 'text'].values,
}).sort_values('novelty', ascending=False).head(5)

for rank, (_, row) in enumerate(ranked.iterrows(), 1):
    text = row['text'][:280]
    print(f"\n  [#{rank}] novelty = {row['novelty']:.3f}")
    print(f"      {text}{'...' if len(row['text']) > 280 else ''}")

SyntaxError: unterminated string literal (detected at line 53) (2075666199.py, line 53)

In [ ]:
# ── Call Claude and parse the response ───────────────────────────
import anthropic
import json

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

def summarize_transition(prev_q: str, cur_q: str, pairs: list) -> dict:
    if len(pairs) == 0:
        return {"themes": [], "error": "no pairs available"}

    user_msg = build_user_message(prev_q, cur_q, pairs)

    response = client.messages.create(
        model=MODEL,
        max_tokens=4000,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_msg}],
    )

    raw = response.content[0].text.strip()

    # Strip markdown code fences if the model wrapped its JSON
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.strip()

    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError as e:
        return {
            "themes": [],
            "error": f"JSON parse failed: {e}",
            "raw_response": raw[:500],
        }

    # Attach usage stats for cost tracking
    parsed['_meta'] = {
        'input_tokens': response.usage.input_tokens,
        'output_tokens': response.usage.output_tokens,
        'pairs_sent': len(pairs),
    }
    return parsed

In [ ]:
# ── Compare LATEST quarter against every prior quarter ───────────
# Uses `quarters_list` from Section 6 (already defined in your notebook).

transition_summaries = {}

cur_q = quarters_list[-1]            # the latest quarter
prior_quarters = quarters_list[:-1]  # everything before it

print(f"Comparing latest quarter {cur_q} against {len(prior_quarters)} prior quarters\n")

for prev_q in prior_quarters:
    print(f"\n{'='*60}")
    print(f"  Analyzing {prev_q} → {cur_q}")
    print(f"{'='*60}")

    pairs = build_transition_payload(df_earnings, cur_q, prev_q)
    if len(pairs) == 0:
        print(f"  (no valid pairs — skipping)")
        continue

    print(f"  Sending {len(pairs)} pairs to Claude...")
    result = summarize_transition(prev_q, cur_q, pairs)
    transition_summaries[f"{prev_q}->{cur_q}"] = result

    if 'error' in result:
        print(f"  ! error: {result['error']}")
        continue

    meta = result.get('_meta', {})
    print(f"  ✓ got {len(result['themes'])} themes"
          f"  ({meta.get('input_tokens', '?')} in / {meta.get('output_tokens', '?')} out tokens)")

Comparing latest quarter FQ2-2026 against 4 prior quarters


  Analyzing FQ2-2025 → FQ2-2026
  Sending 9 pairs to Claude...
  ✓ got 6 themes  (6983 in / 1183 out tokens)

  Analyzing FQ3-2025 → FQ2-2026
  Sending 9 pairs to Claude...
  ✓ got 6 themes  (6983 in / 1016 out tokens)

  Analyzing FQ4-2025 → FQ2-2026
  Sending 9 pairs to Claude...
  ✓ got 6 themes  (6983 in / 1007 out tokens)

  Analyzing FQ1-2026 → FQ2-2026
  Sending 9 pairs to Claude...
  ✓ got 7 themes  (6983 in / 1200 out tokens)


In [ ]:
#pairs = build_transition_payload(df_earnings, cur_q, prev_q)
#if len(pairs) == 0:
#    print(f"  (no valid pairs — skipping)")
#    continue

In [ ]:
# ── Pretty-print the results ─────────────────────────────────────

CATEGORY_ORDER = ['new_product', 'new_segment', 'new_metric',
                  'new_risk', 'new_framing', 'other']

def print_transition(key: str, summary: dict):
    print(f"\n{'═'*72}")
    print(f"  {key}")
    print(f"{'═'*72}")

    if 'error' in summary:
        print(f"  ERROR: {summary['error']}")
        if 'raw_response' in summary:
            print(f"  Raw: {summary['raw_response']}")
        return

    themes = summary.get('themes', [])
    if not themes:
        print("  (no themes identified)")
        return

    # Group by category in consistent order
    by_cat = {}
    for t in themes:
        by_cat.setdefault(t.get('category', 'other'), []).append(t)

    for cat in CATEGORY_ORDER:
        if cat not in by_cat:
            continue
        print(f"\n  {cat.upper()}:")
        for t in by_cat[cat]:
            conf = t.get('confidence', '?')
            label = t.get('label', '(no label)')
            desc = t.get('description', '')
            print(f"    • [{conf}] {label}")
            if desc:
                # wrap at 70 chars, indent continuation lines
                import textwrap
                wrapped = textwrap.fill(desc, width=70,
                                         initial_indent='      ',
                                         subsequent_indent='      ')
                print(wrapped)
            for q in t.get('quotes', [])[:2]:  # show up to 2 quotes
                print(f'        └ "{q[:150]}{"..." if len(q) > 150 else ""}"')
            notes = t.get('notes', '')
            if notes:
                print(f"      note: {notes}")

for key, summary in transition_summaries.items():
    print_transition(key, summary)

    #final output. 


════════════════════════════════════════════════════════════════════════
  FQ2-2025->FQ2-2026
════════════════════════════════════════════════════════════════════════

  NEW_METRIC:
    • [high] OpenAI Investment Impact Reversal to Net Gains
      Microsoft now reports net gains from investments in OpenAI in Q2
      FY2026, marking a reversal from net losses reported in the prior
      quarter and year-ago period. This represents a change from
      reporting losses to gains on these investments.
        └ "In the second quarter of fiscal year 2026, net income and diluted earnings per share were impacted by net gains from investments in OpenAI, which resu..."
        └ "In the second quarter of fiscal year 2025, net income and diluted earnings per share were impacted by net losses from investments in OpenAI, which res..."
      note: This is a substantive change in the accounting treatment or valuation of OpenAI investments, shifting from losses to material gains quarter-over-quarter

In [221]:
# ── Output that is interesting: Print only the most recent quarter transition ──────────────── 

def print_latest_transition(transition_summaries: dict, quarters_list: list):
    """
    Print only the diff between the most recent quarter and the one before it.
    Reuses the existing print_transition() formatter.
    """
    if len(quarters_list) < 2:
        print("  Need at least 2 quarters to show a transition.")
        return
    
    # quarters_list is already chronological (oldest → newest)
    latest_q = quarters_list[-1]
    prior_q  = quarters_list[-2]
    key = f"{prior_q}->{latest_q}"
    
    if key not in transition_summaries:
        print(f"  No summary found for {key}")
        print(f"  Available transitions: {list(transition_summaries.keys())}")
        return
    
    print_transition(key, transition_summaries[key])


# Call it
print_latest_transition(transition_summaries, quarters_list)


════════════════════════════════════════════════════════════════════════
  FQ1-2026->FQ2-2026
════════════════════════════════════════════════════════════════════════

  NEW_METRIC:
    • [high] OpenAI Investment Net Gains vs. Losses
      The current quarter reports net gains from OpenAI investments
      ($7.6 billion increase to net income, $1.02 EPS impact) whereas
      the prior quarter reported net losses ($3.1 billion decrease to
      net income, $0.41 EPS impact). This represents a shift from
      reporting losses to reporting gains on the same investment.
        └ "In the second quarter of fiscal year 2026, net income and diluted earnings per share were impacted by net gains from investments in OpenAI, which resu..."
      note: This is a substantive change in the nature of the impact reported (gains vs. losses), not just different amounts. The prior quarter showed losses of $3.1B/$0.41 EPS.
    • [high] Microsoft Cloud Revenue Crosses $50 Billion Threshold
      The CFO'

In [222]:
# ── Flatten to a dataframe for downstream use ────────────────────
# Useful if you want to filter, export, or chart across quarters.

rows = []
for key, summary in transition_summaries.items():
    if 'error' in summary:
        continue
    prev_q, cur_q = key.split('->')
    for t in summary.get('themes', []):
        rows.append({
            'prev_quarter': prev_q,
            'cur_quarter': cur_q,
            'category': t.get('category'),
            'label': t.get('label'),
            'description': t.get('description'),
            'confidence': t.get('confidence'),
            'n_quotes': len(t.get('quotes', [])),
            'notes': t.get('notes', ''),
        })

df_themes = pd.DataFrame(rows)
print(f"Total themes across all transitions: {len(df_themes)}")
print("\nBy category:")
print(df_themes['category'].value_counts())
print("\nBy transition:")
print(df_themes.groupby('cur_quarter').size())

df_themes.head(20)

Total themes across all transitions: 25

By category:
category
new_metric     16
other           5
new_framing     4
Name: count, dtype: int64

By transition:
cur_quarter
FQ2-2026    25
dtype: int64


,prev_quarter,cur_quarter,category,label,description,confidence,n_quotes,notes
0,FQ2-2025,FQ2-2026,new_metric,OpenAI Investment Impact Reversal to Net Gains,Microsoft now reports net gains from investmen...,high,2,This is a substantive change in the accounting...
1,FQ2-2025,FQ2-2026,new_metric,Microsoft Cloud Revenue Exceeds $50 Billion Mi...,Microsoft now highlights that Microsoft Cloud ...,high,2,While Microsoft Cloud revenue was reported in ...
2,FQ2-2025,FQ2-2026,new_metric,Commercial Remaining Performance Obligation Su...,Microsoft reports commercial remaining perform...,high,1,This represents both a different growth rate (...
3,FQ2-2025,FQ2-2026,new_framing,AI Business Described as Larger Than Major Fra...,The CEO now describes Microsoft's AI business ...,high,2,The prior quarter emphasized 'broad diffusion ...
4,FQ2-2025,FQ2-2026,other,Addition of Conference Call Details,The current quarter adds specific conference c...,medium,2,This appears to be an operational change in ho...
5,FQ2-2025,FQ2-2026,new_metric,Six-Month Financial Data Now Reported,The financial statements now include six-month...,high,2,This is a presentation change that provides ha...
6,FQ3-2025,FQ2-2026,new_metric,Microsoft Cloud $50 billion quarterly revenue ...,Microsoft Cloud revenue crossed $50 billion in...,high,2,Prior quarter reported $49.1 billion; crossing...
7,FQ3-2025,FQ2-2026,new_metric,Commercial remaining performance obligation gr...,Commercial remaining performance obligation in...,high,1,Prior quarter showed 51% growth; the dramatic ...
8,FQ3-2025,FQ2-2026,new_framing,AI business characterized as major franchise,The CEO now describes Microsoft's AI business ...,high,1,Prior quarter emphasized 'planet-scale cloud a...
9,FQ3-2025,FQ2-2026,new_metric,OpenAI investment gains versus prior losses,The company reported net gains of $7.6 billion...,high,2,Prior quarter showed $3.1B loss; this quarter ...


In [223]:
df_themes # aggreagted result in case for future use. 

,prev_quarter,cur_quarter,category,label,description,confidence,n_quotes,notes
0,FQ2-2025,FQ2-2026,new_metric,OpenAI Investment Impact Reversal to Net Gains,Microsoft now reports net gains from investmen...,high,2,This is a substantive change in the accounting...
1,FQ2-2025,FQ2-2026,new_metric,Microsoft Cloud Revenue Exceeds $50 Billion Mi...,Microsoft now highlights that Microsoft Cloud ...,high,2,While Microsoft Cloud revenue was reported in ...
2,FQ2-2025,FQ2-2026,new_metric,Commercial Remaining Performance Obligation Su...,Microsoft reports commercial remaining perform...,high,1,This represents both a different growth rate (...
3,FQ2-2025,FQ2-2026,new_framing,AI Business Described as Larger Than Major Fra...,The CEO now describes Microsoft's AI business ...,high,2,The prior quarter emphasized 'broad diffusion ...
4,FQ2-2025,FQ2-2026,other,Addition of Conference Call Details,The current quarter adds specific conference c...,medium,2,This appears to be an operational change in ho...
5,FQ2-2025,FQ2-2026,new_metric,Six-Month Financial Data Now Reported,The financial statements now include six-month...,high,2,This is a presentation change that provides ha...
6,FQ3-2025,FQ2-2026,new_metric,Microsoft Cloud $50 billion quarterly revenue ...,Microsoft Cloud revenue crossed $50 billion in...,high,2,Prior quarter reported $49.1 billion; crossing...
7,FQ3-2025,FQ2-2026,new_metric,Commercial remaining performance obligation gr...,Commercial remaining performance obligation in...,high,1,Prior quarter showed 51% growth; the dramatic ...
8,FQ3-2025,FQ2-2026,new_framing,AI business characterized as major franchise,The CEO now describes Microsoft's AI business ...,high,1,Prior quarter emphasized 'planet-scale cloud a...
9,FQ3-2025,FQ2-2026,new_metric,OpenAI investment gains versus prior losses,The company reported net gains of $7.6 billion...,high,2,Prior quarter showed $3.1B loss; this quarter ...
